In [12]:
# Cell 1
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from torch.utils.tensorboard import SummaryWriter

In [13]:
# Cell 2
SEED = 42

BATCH_SIZE = 256
EPOCHS = 10
LR = 3e-4
IMG_SIZE = 224

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 2


In [14]:
# Cell 3
GLIOMA_PATHS = [
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing/glioma_tumor",
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Training/glioma_tumor",
    "/kaggle/input/datasets/moundir8/jun-cheng-brain-tumor-mri-dataset-jpg/brain tumor mri dataset/2_glioma"
]

MENINGIOMA_PATHS = [
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing/meningioma_tumor",
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Training/meningioma_tumor",
    "/kaggle/input/datasets/moundir8/jun-cheng-brain-tumor-mri-dataset-jpg/brain tumor mri dataset/1_meningioma"
]

PITUITARY_PATHS = [
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Testing/pituitary_tumor",
    "/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri/Training/pituitary_tumor",
    "/kaggle/input/datasets/moundir8/jun-cheng-brain-tumor-mri-dataset-jpg/brain tumor mri dataset/3_pituitary_tumor"
]

VALID_EXTENSIONS = (".jpg",)

In [15]:
# Cell 4
def collect(paths, label):
    items = []
    for path in paths:
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(VALID_EXTENSIONS):
                    items.append((os.path.join(root, f), label))
    return items

data = (
    collect(GLIOMA_PATHS, 0) +
    collect(MENINGIOMA_PATHS, 1) +
    collect(PITUITARY_PATHS, 2)
)

df = pd.DataFrame(data, columns=["path", "label"])
print("Total Images before deduplication:", len(df))
print("Class Distribution before deduplication:")
print(df["label"].value_counts())

Total Images before deduplication: 5828
Class Distribution before deduplication:
label
0    2352
2    1831
1    1645
Name: count, dtype: int64


In [16]:
# Cell 5
import imagehash

def phash_image(path):
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            return imagehash.phash(img)
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return None

tqdm.pandas(desc="Hashing images (pHash)")
df["hash"] = df["path"].progress_apply(phash_image)

HAMMING_TOLERANCE = 5

unique_indices = []
seen_hashes = []

for idx, h in enumerate(df["hash"]):
    if h is None:
        continue
    if all(h - existing > HAMMING_TOLERANCE for existing in seen_hashes):
        seen_hashes.append(h)
        unique_indices.append(idx)

df = df.loc[unique_indices].reset_index(drop=True)
df = df.drop(columns="hash")

print("Total Images after deduplication:", len(df))
print("Class Distribution after deduplication:")
print(df["label"].value_counts())

Hashing images (pHash):   0%|          | 0/5828 [00:00<?, ?it/s]

Total Images after deduplication: 4705
Class Distribution after deduplication:
label
0    2000
2    1401
1    1304
Name: count, dtype: int64


In [17]:
# Cell 6
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 3293
Val: 706
Test: 706


In [18]:
# Cell 7
class BrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = self.df.loc[idx, "label"]
        
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed loading image: {img_path}") from e
        
        if self.transform:
            img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)

In [19]:
# Cell 8
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [20]:
# Cell 9
train_loader = DataLoader(
    BrainDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=True
)

val_loader = DataLoader(
    BrainDataset(val_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False
)

test_loader = DataLoader(
    BrainDataset(test_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False
)

In [21]:
# Cell 10
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 3)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler("cuda")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 205MB/s]


Using 2 GPUs


In [22]:
# Cell 11
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    loop = tqdm(
        loader,
        total=len(loader),
        desc="Train" if training else "Val/Test",
        leave=False
    )

    for images, labels in loop:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=True):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()
        probs = torch.softmax(outputs, dim=1).detach()
        all_preds.append(probs.cpu())
        all_labels.append(labels.detach().cpu())

        loop.set_postfix(loss=f"{loss.item():.4f}")

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds.argmax(axis=1))
    auc = roc_auc_score(np.eye(3)[all_labels], all_preds, multi_class="ovr")

    return total_loss / len(loader), acc, auc

In [23]:
# Cell 12
log_dir = f"/kaggle/working/runs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir)

best_auc = 0.0

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc, train_auc = run_epoch(train_loader, True)
    val_loss, val_acc, val_auc = run_epoch(val_loader, False)

    scheduler.step()

    print(f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("AUC/Train", train_auc, epoch)
    writer.add_scalar("AUC/Val", val_auc, epoch)

    if val_auc > best_auc:
        best_auc = val_auc
        state_dict = model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
        torch.save(state_dict, "/kaggle/working/best_efficientnet_b0_multi_class.pth")
        print("Best model saved.")

writer.close()


===== Epoch 1/10 =====


Train:   0%|          | 0/12 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/3 [00:00<?, ?it/s]

Train | Loss: 0.6884 | Acc: 0.7266 | AUC: 0.8951
Val   | Loss: 0.7243 | Acc: 0.7139 | AUC: 0.9382
Best model saved.

===== Epoch 2/10 =====


Train:   0%|          | 0/12 [00:00<?, ?it/s]

...

Val/Test:   0%|          | 0/3 [00:00<?, ?it/s]

Train | Loss: 0.0398 | Acc: 0.9889 | AUC: 0.9997
Val   | Loss: 0.1070 | Acc: 0.9547 | AUC: 0.9959
Best model saved.

===== Epoch 10/10 =====


Train:   0%|          | 0/12 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/3 [00:00<?, ?it/s]

Train | Loss: 0.0457 | Acc: 0.9854 | AUC: 0.9995
Val   | Loss: 0.1078 | Acc: 0.9561 | AUC: 0.9958


In [24]:
# Cell 13
state_dict = torch.load("/kaggle/working/best_efficientnet_b0_multi_class.pth")

if torch.cuda.device_count() > 1:
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

test_loss, test_acc, test_auc = run_epoch(test_loader, training=False)

print("\n===== FINAL UNSEEN TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Val/Test:   0%|          | 0/3 [00:00<?, ?it/s]


===== FINAL UNSEEN TEST RESULTS =====
Test Loss: 0.0582
Test Accuracy: 0.9759
Test AUC: 0.9987


In [25]:
# Cell 14
final_state = model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
torch.save(final_state, "/kaggle/working/final_efficientnet_b0_brain_tumor_multi_class.pth")

print("Final model saved.")

Final model saved.
